In [71]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [72]:
from utils import load_measurement_data
from joystick import make_dataset, setup_magnets, setup_sensor
from parameters import calibration_values, magnetization_values, parameter_factory
import matplotlib.pyplot as plt
import numpy as np
from sklearn.decomposition import PCA
import magpylib as mpl
import pandas as pd
from sklearn.model_selection import train_test_split

In [73]:
data_meas = load_measurement_data("data/sensor1")

In [74]:
def plot_loops(B, tilt=0):
    fig = plt.figure(2, figsize=(4, 4))
    ax = fig.add_subplot(111, projection="3d")
    ax.plot(B[:, 0], B[:, 1], B[:, 2], label="tilt=south")
    ax.set_xlabel("$B_x$ [mT]")
    ax.set_ylabel("$B_y$ [mT]")
    ax.set_zlabel("$B_z$ [mT]")
    ax.legend(frameon=False)
    ax.view_init(elev=20, azim=-77)  # tweak azim until labels clear the edge
    plt.tight_layout(pad=0.2)
    plt.show()

In [75]:
sensor = setup_sensor(parameters=parameter_factory())
magnets = setup_magnets(
    parameters=parameter_factory(),
    magnetizations=magnetization_values(),
)


In [80]:
X, y = make_dataset(
    calibration=calibration_values(),
    magnetizations=magnetization_values(),
    seed=1,
    n_steps=24,
)

X

[make_dataset]: 0.006335s


,Bx,By,Bz
0,-0.005614,0.005023,0.004442
1,-0.003750,-0.000694,-0.000854
2,-0.007298,-0.012968,-0.003930
3,-0.046210,-0.061246,0.033335
4,-0.082699,0.028057,0.068431
...,...,...,...
115,0.003537,0.004732,-0.003541
116,0.008718,0.009559,-0.016241
117,0.017677,-0.022419,-0.033130
118,-0.015315,-0.021144,0.031629


In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, train_size=0.75)

In [68]:
sens = mpl.Sensor()
sens = sens.rotate_from_euler((10, 10, 10), seq=("xyz"), anchor=None)
sens.orientation.as_matrix()

array([[ 0.96984631, -0.14131448,  0.19856573],
       [ 0.17101007,  0.97508244, -0.14131448],
       [-0.17364818,  0.17101007,  0.96984631]])

In [12]:
data = X.copy()
data["tilt"] = y["tilt"].astype(np.int8)
data["angle_idx"] = (y["angle"] / 15).astype(np.int16)

tilt_ref = {0: "ground", 1: "south", 2: "north", 3: "west", 4: "east"}

data_heirarchical = {}
for k, v in tilt_ref.items():
    tilt_state = data[data["tilt"] == k][["Bx", "By", "Bz"]]
    angles = data[data["tilt"] == k]["angle_idx"]
    data_heirarchical[v] = tilt_state.to_numpy()
    data_heirarchical[v] = angles

data_heirarchical

{'ground': 0      0
 1      1
 2      2
 3      3
 4      4
 5      5
 6      6
 7      7
 8      8
 9      9
 10    10
 11    11
 12    12
 13    13
 14    14
 15    15
 16    16
 17    17
 18    18
 19    19
 20    20
 21    21
 22    22
 23    23
 Name: angle_idx, dtype: int16,
 'south': 24     0
 25     1
 26     2
 27     3
 28     4
 29     5
 30     6
 31     7
 32     8
 33     9
 34    10
 35    11
 36    12
 37    13
 38    14
 39    15
 40    16
 41    17
 42    18
 43    19
 44    20
 45    21
 46    22
 47    23
 Name: angle_idx, dtype: int16,
 'north': 48     0
 49     1
 50     2
 51     3
 52     4
 53     5
 54     6
 55     7
 56     8
 57     9
 58    10
 59    11
 60    12
 61    13
 62    14
 63    15
 64    16
 65    17
 66    18
 67    19
 68    20
 69    21
 70    22
 71    23
 Name: angle_idx, dtype: int16,
 'west': 72     0
 73     1
 74     2
 75     3
 76     4
 77     5
 78     6
 79     7
 80     8
 81     9
 82    10
 83    11
 84    12
 85    13
 86    1

In [16]:
K_ANGLES = 24  # 360° / 15°

# tilt codes: 0 = flat, 1/2/3/4 = S/N/W/E
_TILT_NAMES = {1: "south", 2: "north", 3: "west", 4: "east"}


def _tilt_transition(start_tilt, end_tilt):
    """Name the tilt move, or None if the pair isn't a legal transition.

    Legal moves: staying put, flat -> tilted, or tilted -> flat.
    Tilt-to-tilt (e.g. N -> S) is illegal.
    """
    if start_tilt == end_tilt:
        return "noop"
    if start_tilt == 0:  # raising into a tilt
        return f"{_TILT_NAMES[end_tilt]}"
    if end_tilt == 0:  # lowering back to flat
        return f"{_TILT_NAMES[start_tilt]}"
    return None


def _rot_transition(start_idx, end_idx, k=K_ANGLES):
    """Name the rotation move, or None if start/end aren't adjacent.

    step = (end - start) mod k. Increasing index is CW, so a +1 notch
    is clockwise.
    """
    step = (int(end_idx) - int(start_idx)) % k
    if step == 0:
        return "noop"
    if step == 1:
        return "cw"  # +1 notch (e.g. 2 -> 3)
    if step == k - 1:
        return "ccw"  # -1 notch
    return None


def to_state(state):
    return {
        "rotation": state["angle_idx"] * 15.0,
        "tilt": state["tilt"],
        "B": np.asarray((state["Bx"], state["By"], state["Bz"])).tolist(),
    }


def make_pairs(X, y):
    data = X.copy()
    data["tilt"] = y["tilt"].astype(np.int8)
    data["angle_idx"] = (y["angle"] / 15).astype(np.int16)

    transitions = []
    for _, start in data.iterrows():
        for _, end in data.iterrows():
            tilt_move = _tilt_transition(start["tilt"], end["tilt"])
            rot_move = _rot_transition(start["angle_idx"], end["angle_idx"])
            if tilt_move is not None and rot_move is not None:
                info = {
                    "tilt_direction": tilt_move,
                    "rot_direction": rot_move,
                    "start": to_state(start),
                    "end": to_state(end),
                }
                transitions.append(info)

    return transitions


In [17]:
import json


def save_json(path, transitions):

    with open(path, "w+") as f:
        json.dump(transitions, f)

In [18]:
transitions = make_pairs(X, y)


save_json("traj.json", transitions)